In [20]:
# 1: Install the required libraries

!pip -q install scikit-learn pandas gradio

In [21]:
# 2: Download and extract email data

import os
import tarfile
import urllib.request
import pandas as pd
from email import policy
from email.parser import BytesParser
from IPython.display import display

BASE_URL = "https://spamassassin.apache.org/old/publiccorpus/"

archives = {
    "20030228_easy_ham.tar.bz2": "ham",
    "20030228_easy_ham_2.tar.bz2": "ham",
    "20030228_spam.tar.bz2": "spam",
    "20030228_spam_2.tar.bz2": "spam"
}

os.makedirs("email_dataset", exist_ok=True)

for filename in archives:
    filepath = os.path.join("email_dataset", filename)

    if not os.path.exists(filepath):
        print("Downloading:", filename)
        urllib.request.urlretrieve(BASE_URL + filename, filepath)

    with tarfile.open(filepath, "r:bz2") as archive:
        archive.extractall("email_dataset")

print("\nAll four email archives downloaded and extracted!")

Downloading: 20030228_easy_ham.tar.bz2


/tmp/ipykernel_1647/3094441553.py:30: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  archive.extractall("email_dataset")


Downloading: 20030228_easy_ham_2.tar.bz2
Downloading: 20030228_spam.tar.bz2
Downloading: 20030228_spam_2.tar.bz2

All four email archives downloaded and extracted!


In [24]:
# 3: Load and parse the extracted email dataset

import os
import pandas as pd
from email import policy
from email.parser import BytesParser
from IPython.display import display

emails = []

# These are the actual folders in your Colab output
email_folders = {
    "email_dataset/easy_ham": "ham",
    "email_dataset/easy_ham_2": "ham",
    "email_dataset/spam": "spam",
    "email_dataset/spam_2": "spam"
}

def extract_email_text(filepath):
    try:
        with open(filepath, "rb") as file:
            msg = BytesParser(policy=policy.default).parse(file)

        subject = str(msg.get("subject", "") or "")
        body_parts = []

        parts = msg.walk() if msg.is_multipart() else [msg]

        for part in parts:
            if part.get_content_type() != "text/plain":
                continue

            if part.get_content_disposition() == "attachment":
                continue

            try:
                content = part.get_content()
                if isinstance(content, str):
                    body_parts.append(content)
            except Exception:
                pass

        body = "\n".join(body_parts)
        return f"Subject: {subject}\n\n{body}".strip()

    except Exception:
        return ""

for folder, label in email_folders.items():
    if not os.path.isdir(folder):
        print("Folder not found:", folder)
        continue

    for filename in os.listdir(folder):
        filepath = os.path.join(folder, filename)

        if not os.path.isfile(filepath):
            continue

        # Ignore non-email files such as helper files
        if "." not in filename or filename.startswith("."):
            continue

        message = extract_email_text(filepath)

        if message:
            emails.append({
                "label": label,
                "message": message
            })

df = pd.DataFrame(emails, columns=["label", "message"])

df = df.drop_duplicates(subset="message").reset_index(drop=True)
df = df[df["message"].str.strip() != ""].reset_index(drop=True)

print("Email dataset loaded successfully!")
print("Dataset shape:", df.shape)
print("\nClass distribution:")
print(df["label"].value_counts())

display(df.head())

Email dataset loaded successfully!
Dataset shape: (5647, 2)

Class distribution:
label
ham     3861
spam    1786
Name: count, dtype: int64


,label,message
0,ham,Subject: [SAtalk] Spamassassin with Pyzor\n\nI...
1,ham,Subject: Re[2]: Selling Wedded Bliss (was Re: ...
2,ham,Subject: Re: [SAtalk] Highest-scoring false po...
3,ham,Subject: [SAtalk] RBL timed oiut and Spam Assa...
4,ham,Subject: 'Seminal' eczema study reveals skin's...


In [26]:
# 4: Split data into training and testing sets

from sklearn.model_selection import train_test_split

X = df["message"].astype(str)
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training emails:", len(X_train))
print("Testing emails:", len(X_test))
print("Data split completed!")

Training emails: 4517
Testing emails: 1130
Data split completed!


In [27]:
#  5: Build and train the ML model

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.naive_bayes import ComplementNB

text_features = FeatureUnion([
    (
        "word_features",
        TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            max_features=150000,
            sublinear_tf=True
        )
    ),
    (
        "character_features",
        TfidfVectorizer(
            analyzer="char",
            ngram_range=(3, 5),
            min_df=2,
            max_features=150000,
            sublinear_tf=True
        )
    )
])

spam_model = Pipeline([
    ("features", text_features),
    ("classifier", ComplementNB(alpha=0.3))
])

spam_model.fit(X_train, y_train)

print("Email spam detection model trained successfully!")

Email spam detection model trained successfully!


In [28]:
# 6: Evaluate performance on unseen test emails

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

y_pred = spam_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred) * 100

print(f"Test accuracy: {accuracy:.2f}%")
print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Legitimate (Ham)", "Spam"]
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Test accuracy: 97.08%

Classification Report:
                  precision    recall  f1-score   support

Legitimate (Ham)       0.96      1.00      0.98       773
            Spam       1.00      0.91      0.95       357

        accuracy                           0.97      1130
       macro avg       0.98      0.95      0.97      1130
    weighted avg       0.97      0.97      0.97      1130

Confusion Matrix:
[[772   1]
 [ 32 325]]


In [29]:
# 7: Predict whether an email is spam

def check_email(subject, body):
    subject = subject or ""
    body = body or ""

    if not subject.strip() and not body.strip():
        return "Please enter an email subject or body."

    email_text = f"Subject: {subject}\n\n{body}"

    prediction = spam_model.predict([email_text])[0]
    probabilities = spam_model.predict_proba([email_text])[0]

    classes = list(spam_model.named_steps["classifier"].classes_)
    spam_index = classes.index("spam")
    spam_probability = probabilities[spam_index] * 100

    if prediction == "spam":
        result = "🚨 SPAM EMAIL"
        explanation = "This email contains patterns associated with spam in the training data."
    else:
        result = "✅ LEGITIMATE EMAIL"
        explanation = "This email was classified as legitimate by the model."

    return (
        f"### {result}\n\n"
        f"**Estimated spam probability:** {spam_probability:.2f}%\n\n"
        f"{explanation}\n\n"
        f"**Subject:** {subject or '(No subject)'}\n\n"
        f"**Note:** This is an ML prediction, not a guarantee."
    )

print("Email prediction function ready!")

Email prediction function ready!


In [34]:
# Cell 8: Test the model on sample emails

test_emails = [
    {
        "type": "Spam",
        "subject": "Congratulations! You won a prize!",
        "body": "You have won a free iPhone. Click now to claim your prize."
    },
    {
        "type": "Spam",
        "subject": "URGENT: Your account is suspended",
        "body": "Verify your password immediately to avoid account suspension."
    },
    {
        "type": "Spam",
        "subject": "Claim your lottery winnings",
        "body": "You won a lottery prize. Send your bank details to receive the money."
    },
    {
        "type": "Spam",
        "subject": "Work from home and earn money",
        "body": "Earn thousands every day. Pay a registration fee to start earning."
    },
    {
        "type": "Spam",
        "subject": "Free gift card waiting for you",
        "body": "Claim your free gift card now before this offer expires."
    },
    {
        "type": "Legitimate",
        "subject": "Project meeting tomorrow",
        "body": "Hi team, our project meeting is scheduled for 11 AM tomorrow."
    },
    {
        "type": "Legitimate",
        "subject": "Assignment submission reminder",
        "body": "Please submit your assignment by Friday. Contact me with questions."
    },
    {
        "type": "Legitimate",
        "subject": "Meeting notes",
        "body": "Hi everyone, attached are the notes from today's meeting. Thank you."
    },
    {
        "type": "Legitimate",
        "subject": "Lunch plans",
        "body": "Hey, are you free to have lunch today? Let me know what time works."
    },
    {
        "type": "Legitimate",
        "subject": "Weekly project update",
        "body": "We completed the first project milestone. Our next task is testing."
    }
]

results = []

for email in test_emails:
    email_text = f"Subject: {email['subject']}\n\n{email['body']}"

    prediction = spam_model.predict([email_text])[0]
    probabilities = spam_model.predict_proba([email_text])[0]

    classes = list(spam_model.named_steps["classifier"].classes_)
    spam_probability = probabilities[classes.index("spam")] * 100

    predicted_label = "Spam" if prediction == "spam" else "Legitimate"

    results.append({
        "Expected": email["type"],
        "Subject": email["subject"],
        "Predicted": predicted_label,
        "Spam Probability (%)": round(spam_probability, 2),
        "Correct?": email["type"] == predicted_label
    })

results_df = pd.DataFrame(results)
display(results_df)

print(
    f"\nCorrect predictions: "
    f"{results_df['Correct?'].sum()} out of {len(results_df)}"
)

,Expected,Subject,Predicted,Spam Probability (%),Correct?
0,Spam,Congratulations! You won a prize!,Spam,99.92,True
1,Spam,URGENT: Your account is suspended,Spam,99.33,True
2,Spam,Claim your lottery winnings,Spam,99.90,True
3,Spam,Work from home and earn money,Spam,99.97,True
4,Spam,Free gift card waiting for you,Spam,100.00,True
5,Legitimate,Project meeting tomorrow,Legitimate,21.49,True
6,Legitimate,Assignment submission reminder,Spam,97.32,False
7,Legitimate,Meeting notes,Legitimate,43.28,True
8,Legitimate,Lunch plans,Legitimate,14.58,True
9,Legitimate,Weekly project update,Legitimate,2.55,True



Correct predictions: 9 out of 10


In [35]:
#  9: Create the Gradio dashboard

import gradio as gr

custom_css = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&display=swap');

.gradio-container,
.gradio-container * {
    font-family: 'Inter', sans-serif !important;
}

.gradio-container {
    max-width: 1000px !important;
    margin: auto !important;
}

#main-title {
    text-align: center;
}

#subtitle {
    text-align: center;
    opacity: 0.8;
}
"""

with gr.Blocks(title="SpamShield AI") as app:

    gr.Markdown(
        """
        # 🛡️ SpamShield AI
        ### Intelligent Email Spam Detection
        Enter an email subject and its body to analyze it using machine learning.
        """,
        elem_id="main-title"
    )

    gr.Markdown(
        "AI-powered email classification using TF-IDF and Complement Naive Bayes.",
        elem_id="subtitle"
    )

    with gr.Row():
        with gr.Column():
            subject_input = gr.Textbox(
                label="📩 Email Subject",
                placeholder="Enter the email subject...",
                lines=2
            )

            body_input = gr.Textbox(
                label="📝 Email Body",
                placeholder="Paste the email content here...",
                lines=10
            )

            with gr.Row():
                analyze_button = gr.Button(
                    "🔍 Analyze Email",
                    variant="primary"
                )
                clear_button = gr.ClearButton(
                    [subject_input, body_input]
                )

        with gr.Column():
            output = gr.Markdown(
                label="Analysis Result",
                value="### Your result will appear here\n\nEnter an email and click **Analyze Email**."
            )

    analyze_button.click(
        fn=check_email,
        inputs=[subject_input, body_input],
        outputs=output
    )

    gr.Markdown(
        """
        ---
        **How it works:** The model converts email text into numerical
        TF-IDF features and uses a trained classifier to predict spam or
        legitimate email.

        **Disclaimer:** Predictions can be wrong. Avoid relying on this
        prototype as the sole filter for important emails.
        """
    )

app.launch(
    share=True,
    theme=gr.themes.Soft(),
    css=custom_css
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b43dae7e9f91e34864.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
